# Mitigasi D: menambah prediktor merokok (dan membiarkan BMI berbeda makna menurut jenis kelamin)
Lanjutan `audit_kesetaraan_hsys.ipynb`. Pertanyaan: bila laki-laki yang meradang banyak yang kurus,
apakah menambah informasi **merokok** mengurangi kasus laki-laki yang terlewat **tanpa** menaikkan kasus perempuan yang terlewat?
Model dibandingkan pada prosedur yang sama (CV 5-fold × 3 per rumah tangga + Platt di dalam fold), ambang 0,155.
Waktu: ±5–10 menit.

In [ ]:
import os, time, warnings
import numpy as np, pandas as pd
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
warnings.filterwarnings("ignore")

ROOT = r"F:\SINTA 3\crp-ifls5-ml"
OUT = os.path.join(ROOT, "results")
df = pd.read_csv(os.path.join(ROOT, "data", "crp_ifls5_model.csv"), dtype={"pidlink": str, "hhid14": str})
SEED, REP, THR, NB = 42, 3, 0.155, 300
y, g = df.crp_high.values.astype(int), df.hhid14.values
rng = np.random.default_rng(SEED)

print("Isi smoke_status:", df.smoke_status.value_counts(dropna=False).sort_index().to_dict())
ss = pd.to_numeric(df.smoke_status, errors="coerce")
# kode build14: 0 = tidak pernah, 1 = mantan, 2 = merokok sekarang
df["current_smoker"] = (ss == ss.max()).astype(float).where(ss.notna())
df["ever_smoker"] = (ss > ss.min()).astype(float).where(ss.notna())
df["cig_day"] = pd.to_numeric(df.cig_day, errors="coerce").clip(0, 80)
df["male"] = 1 - df.female
df["male_x_bmi"] = df.male * df.bmi
df["male_x_smoker"] = df.male * df.current_smoker
print("Perokok aktif: laki-laki", f"{df.current_smoker[df.male == 1].mean()*100:.1f}%",
      "| perempuan", f"{df.current_smoker[df.female == 1].mean()*100:.1f}%")
print("hs-CRP tinggi pada laki-laki: perokok", f"{y[(df.male == 1) & (df.current_smoker == 1)].mean()*100:.1f}%",
      "vs bukan perokok", f"{y[(df.male == 1) & (df.current_smoker == 0)].mean()*100:.1f}%")

FIVE = ["bmi", "female", "age", "srh_poor", "urban"]
SPECS = {
    "Five-item rule (national)": FIVE,
    "D1. Five items + smoking": FIVE + ["current_smoker", "cig_day"],
    "D2. D1 + sex-specific BMI and smoking": FIVE + ["current_smoker", "cig_day", "male_x_bmi", "male_x_smoker"],
}

In [ ]:
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))
def model():
    return Pipeline([("imp", SimpleImputer(strategy="median")), ("sc", StandardScaler()),
                     ("m", LogisticRegression(class_weight="balanced", max_iter=3000))])
def platt(p, yt):
    lr = LogisticRegression(C=1e6, max_iter=1000).fit(logit(p).reshape(-1, 1), yt)
    return lambda s: lr.predict_proba(logit(s).reshape(-1, 1))[:, 1]

def oof(cols):
    X = df[cols].astype(float); P = np.zeros((REP, len(y)))
    for r in range(REP):
        for tr, te in StratifiedGroupKFold(5, shuffle=True, random_state=SEED + r).split(X, y, g):
            m = model().fit(X.iloc[tr], y[tr]); pin = np.zeros(len(tr))
            for a, b in StratifiedGroupKFold(3, shuffle=True, random_state=SEED).split(X.iloc[tr], y[tr], g[tr]):
                pin[b] = model().fit(X.iloc[tr].iloc[a], y[tr][a]).predict_proba(X.iloc[tr].iloc[b])[:, 1]
            P[r, te] = platt(pin, y[tr])(m.predict_proba(X.iloc[te])[:, 1])
    return P.mean(0)

t0 = time.time(); PRED = {}
for k, c in SPECS.items():
    PRED[k] = oof(c); print(f"{k}: selesai ({(time.time()-t0)/60:.1f} menit)")

In [ ]:
sex = np.where(df.female == 1, "Female", "Male")
res = np.where(df.urban == 1, "Urban", "Rural")
REGION = {1: "Sumatra", 3: "Java", 5: "Bali–Nusa Tenggara", 6: "Kalimantan", 7: "Sulawesi"}
reg = (pd.to_numeric(df.province, errors="coerce") // 10).map(REGION).fillna("Other").values
HH = pd.Series(g); uniq = HH.unique(); ix = HH.groupby(HH).indices
BOOT = [np.concatenate([ix[h] for h in rng.choice(uniq, len(uniq))]) for _ in range(NB)]

def fnr(p, m):
    m = m & (y == 1); return (p[m] < THR).mean() * 100

rows = []
for k, p in PRED.items():
    f = p >= THR
    def gap(b):
        pm, pf = p[b][(sex[b] == "Male") & (y[b] == 1)], p[b][(sex[b] == "Female") & (y[b] == 1)]
        return ((pm < THR).mean() - (pf < THR).mean()) * 100
    bs = np.array([gap(b) for b in BOOT])
    row = {"Model": k, "AUC": roc_auc_score(y, p), "O/E": y.mean() / p.mean(),
           "Flagged (%)": f.mean() * 100, "Sensitivity": (f & (y == 1)).sum() / y.sum(),
           "Specificity": (~f & (y == 0)).sum() / (y == 0).sum(),
           "AUC men": roc_auc_score(y[sex == "Male"], p[sex == "Male"]),
           "AUC women": roc_auc_score(y[sex == "Female"], p[sex == "Female"]),
           "FNR men (%)": fnr(p, sex == "Male"), "FNR women (%)": fnr(p, sex == "Female"),
           "FNR gap M−F (pp)": gap(np.arange(len(y))),
           "Gap 95% CI": f"{np.percentile(bs, 2.5):.1f} to {np.percentile(bs, 97.5):.1f}",
           "FNR rural (%)": fnr(p, res == "Rural"), "FNR urban (%)": fnr(p, res == "Urban")}
    fr = {rg: fnr(p, reg == rg) for rg in REGION.values()}
    row.update({f"FNR {rg} (%)": v for rg, v in fr.items()})
    row["FNR range across regions (pp)"] = max(fr.values()) - min(fr.values())
    rows.append(row)
MITD = pd.DataFrame(rows).set_index("Model")
pd.set_option("display.width", 250)
print(MITD.round(3).T.to_string())

# koefisien model D2 (dilatih pada semua data, untuk tafsiran; bukan untuk evaluasi)
m = model().fit(df[SPECS["D2. D1 + sex-specific BMI and smoking"]].astype(float), y)
coef = pd.Series(m[-1].coef_[0], index=SPECS["D2. D1 + sex-specific BMI and smoking"]).round(3)
print("\nKoefisien terstandar model D2:\n", coef.to_string())

with pd.ExcelWriter(os.path.join(OUT, "audit_mitigasi_rokok.xlsx")) as w:
    MITD.round(4).to_excel(w, sheet_name="mitigasi_D")
    coef.to_frame("std_coef").to_excel(w, sheet_name="koefisien_D2")
print("Tersimpan:", os.path.join(OUT, "audit_mitigasi_rokok.xlsx"), "| Simpan notebook (Ctrl+S) lalu kabari Claude.")